# Regression Homework 2026



## Preparing the dataset

Use only the following columns:

'engine_displacement',
'horsepower',
'vehicle_weight',
'model_year',
'fuel_efficiency_mpg'

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

data = pd.read_csv("../data/car_fuel_efficiency_2026.csv")

cols = ['engine_displacement', 'horsepower', 'vehicle_weight',
        'model_year', 'fuel_efficiency_mpg']
df = data[cols]
print(df.head())

## EDA

Look at the fuel_efficiency_mpg variable. Does it have a long tail?

In [ ]:
sns.histplot(df['fuel_efficiency_mpg'], bins=50)
plt.show()

print("Skewness of fuel_efficiency_mpg:", df['fuel_efficiency_mpg'].skew())
print("Mean of fuel_efficiency_mpg:", df['fuel_efficiency_mpg'].mean())
print("Median of fuel_efficiency_mpg:", df['fuel_efficiency_mpg'].median())

# df['fuel_efficiency_mpg'].describe()
df['fuel_efficiency_mpg'].quantile([0.01, 0.05, 0.5, 0.95, 0.99])

No, it doesn't have a long tail. The distribution is roughly bell-shaped (normal) and symmetric.
* Skewness is about 0.08, very close to 0. A long-tailed distribution would show a clearly larger value.
* The mean (≈30.0) and median (30.0) are essentially identical.
* The values range from 19.8 to 41.2 and thin out evenly on both sides. The 1st percentile is 23.3 and the 99th is 36.9, about 6.7 below and 6.9 above the median.


## 1. Missing column

In [ ]:
print(df.isna().sum())

'horsepower' has missing values (877). All the other columns have 0.

## 2. Median of 'horsepower'?

In [ ]:
print("Median of horsepower:", df['horsepower'].median())

## Prepare and split the dataset

In [ ]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

print(len(df_train), len(df_val), len(df_test))   # 6000 2000 2000

## 3. Model Training

In [ ]:
# Define features and target variable
features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

# function to train linear regression model with optional regularization
def train_linear_regression(X, y, r=0.0):
    X = np.column_stack([np.ones(X.shape[0]), X])   # add bias column
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])            # r=0 means no regularization
    w = np.linalg.inv(XTX).dot(X.T).dot(y)
    return w[0], w[1:]

def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())

def prepare_X(df, fill_value):
    return df[features].fillna(fill_value).values

In [ ]:
mean_hp = df_train['horsepower'].mean()    # training data only

# run regression with both fill values: 0 and mean of horsepower
for name, fill in [('0', 0), ('mean', mean_hp)]:
    w0, w = train_linear_regression(prepare_X(df_train, fill), y_train)
    y_pred = w0 + prepare_X(df_val, fill).dot(w)
    print(name, round(rmse(y_val, y_pred), 3))

Answer: With mean horspower we get a lower RMSE (2.202 vs 2.205).

## 4. Regularized linear regression

In [ ]:
# NAs are filled with 0
X_train = prepare_X(df_train, 0)
X_val = prepare_X(df_val, 0)

# run regression with different regularization parameters
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression(X_train, y_train, r)
    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)
    print(f"{r:>6} | {score:>8.4f}")

Answer: 0. Regularization only makes the validation RMSE worse here, so the best value is no regularization.

## 5. Assess seed selection impact on RMSE

In [ ]:
scores = []

for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]

    X_train = prepare_X(df_train, 0)      # fill NAs with 0
    X_val = prepare_X(df_val, 0)
    y_train = df_train['fuel_efficiency_mpg'].values
    y_val = df_val['fuel_efficiency_mpg'].values

    w0, w = train_linear_regression(X_train, y_train)    # r=0, no regularization
    score = rmse(y_val, w0 + X_val.dot(w))
    scores.append(score)

print(round(np.std(scores), 3))    # 0.029

## 6. 

In [ ]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

# Combine train and validation
df_full_train = pd.concat([df_train, df_val])

X_full_train = prepare_X(df_full_train, 0)      # fill NAs with 0
y_full_train = df_full_train['fuel_efficiency_mpg'].values

w0, w = train_linear_regression(X_full_train, y_full_train, r=0.001)

X_test = prepare_X(df_test, 0)
y_test = df_test['fuel_efficiency_mpg'].values

print(rmse(y_test, w0 + X_test.dot(w)))    # 2.2358